# Block Bids & Linked Block Bids — Borgholt CCGT Plant (DE-LU)

**Scenario:** Borgholt Energie GmbH operates a 400 MW Combined-Cycle Gas Turbine (CCGT) plant
in the DE-LU bidding zone. Starting a CCGT has a significant fixed cost (~€15,000) which must
be recovered. The trading desk wants to model:

1. A **startup cost block bid** (linked block) covering the 2-hour startup period
2. A **main generation block bid** for 6 hours of operation at 350 MW
3. An **exclusive group bid** offering two operating modes (base-load vs. peak-load)

This notebook demonstrates:
- `BlockBid` for all-or-nothing blocks
- `LinkedBlockBid` to model startup cost dependency
- `ExclusiveGroupBid` for mutually exclusive operating modes
- Gantt-chart visualisation of the bid timeline

## Prerequisites

```bash
pip install nexa-bidkit matplotlib
# or, from source:
poetry install
```

In [ ]:
from datetime import datetime, timedelta, timezone
from decimal import Decimal
import zoneinfo

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from nexa_bidkit.bids import (
    block_bid,
    exclusive_group,
    indivisible_block_bid,
    linked_block_bid,
)
from nexa_bidkit.types import (
    BiddingZone,
    DeliveryPeriod,
    Direction,
    MTUDuration,
)

CET = zoneinfo.ZoneInfo("Europe/Berlin")
print("nexa-bidkit imports OK")

## 1. Delivery Periods

CCGT economics require multi-hour commitment blocks. We define three delivery periods:

| Block | Period | Purpose |
|-------|--------|---------|
| Startup | 06:00–08:00 | Startup/ramp-up cost recovery |
| Base-load | 08:00–14:00 | 6h generation at 350 MW, EUR 65/MWh |
| Peak-load | 08:00–12:00 | 4h generation at 400 MW, EUR 85/MWh |

In [ ]:
# Delivery day: 2026-03-15
delivery_day = datetime(2026, 3, 15, tzinfo=CET)
MTU = MTUDuration.QUARTER_HOURLY  # 15-min MTUs

def make_period(hour_start: int, hour_end: int) -> DeliveryPeriod:
    """Build a DeliveryPeriod from hour offsets on the delivery day."""
    return DeliveryPeriod(
        start=delivery_day + timedelta(hours=hour_start),
        end=delivery_day + timedelta(hours=hour_end),
        duration=MTU,
    )

startup_period = make_period(6, 8)     # 06:00–08:00 (8 MTUs)
base_load_period = make_period(8, 14)  # 08:00–14:00 (24 MTUs)
peak_load_period = make_period(8, 12)  # 08:00–12:00 (16 MTUs)

print(f"Startup period:    {startup_period.mtu_count} MTUs ({startup_period.start.astimezone(CET).strftime('%H:%M')}–{startup_period.end.astimezone(CET).strftime('%H:%M')} CET)")
print(f"Base-load period:  {base_load_period.mtu_count} MTUs ({base_load_period.start.astimezone(CET).strftime('%H:%M')}–{base_load_period.end.astimezone(CET).strftime('%H:%M')} CET)")
print(f"Peak-load period:  {peak_load_period.mtu_count} MTUs ({peak_load_period.start.astimezone(CET).strftime('%H:%M')}–{peak_load_period.end.astimezone(CET).strftime('%H:%M')} CET)")

## 2. Main Generation Block Bid

The main generation block represents 6 hours at 350 MW for EUR 65/MWh.
It must be accepted in full (`min_acceptance_ratio=1.0`), so the plant only
runs if the full 6-hour block is cleared.

In [ ]:
main_block = indivisible_block_bid(
    bidding_zone=BiddingZone.DE_LU,
    direction=Direction.SELL,
    delivery_period=base_load_period,
    price=Decimal("65.00"),   # EUR/MWh minimum acceptable price
    volume=Decimal("350"),     # MW per MTU
    bid_id="borgholt-main-block",
    metadata={"asset": "borgholt-ccgt", "bid_strategy": "base-load"},
)

print(f"Main block bid:")
print(f"  ID:           {main_block.bid_id}")
print(f"  Price:        {main_block.price} EUR/MWh")
print(f"  Volume/MTU:   {main_block.volume} MW")
print(f"  Total volume: {main_block.total_volume} MW·MTUs")
print(f"  Indivisible:  {main_block.is_indivisible}")
print(f"  MTUs:         {main_block.delivery_period.mtu_count}")

## 3. Linked Block Bid — Startup Cost Recovery

The startup cost (€15,000 spread over 2 hours = €21.43/MWh at 350 MW) is submitted as a
**linked block bid**. It is only accepted if the parent `main_block` is accepted — preventing
the plant from incurring startup costs without receiving generation revenue.

In [ ]:
# Startup cost calculation
STARTUP_COST_EUR = Decimal("15000")
startup_mtu_count = startup_period.mtu_count  # 8 MTUs × 15 min = 2 hours
startup_volume_mw = Decimal("350")  # MW during ramp-up (partial output)

# Total MWh during startup (hours × MW, but MWh = MW × hours)
startup_hours = Decimal(str(startup_mtu_count)) * Decimal("0.25")  # quarter hours to hours
startup_mwh = startup_volume_mw * startup_hours  # 350 × 2 = 700 MWh

# Minimum price to recover startup cost on top of marginal cost (EUR 55/MWh)
startup_marginal = Decimal("55.00")
startup_recovery_price = startup_marginal + (STARTUP_COST_EUR / startup_mwh)

print(f"Startup period:  {startup_hours} hours, {startup_mwh} MWh at {startup_volume_mw} MW")
print(f"Cost to recover: EUR {STARTUP_COST_EUR}")
print(f"Startup price:   EUR {startup_recovery_price:.2f}/MWh (marginal + recovery)")

startup_linked = linked_block_bid(
    parent_bid_id="borgholt-main-block",  # only accepted if main block is accepted
    bidding_zone=BiddingZone.DE_LU,
    direction=Direction.SELL,
    delivery_period=startup_period,
    price=startup_recovery_price.quantize(Decimal("0.01")),
    volume=startup_volume_mw,
    bid_id="borgholt-startup",
    metadata={"asset": "borgholt-ccgt", "cost_type": "startup"},
)

print(f"\nLinked block bid:")
print(f"  ID:          {startup_linked.bid_id}")
print(f"  Parent:      {startup_linked.parent_bid_id}")
print(f"  Price:       {startup_linked.price} EUR/MWh")
print(f"  Volume/MTU:  {startup_linked.volume} MW")

## 4. Exclusive Group Bid — Two Operating Modes

The CCGT can operate in two modes. At most one mode can be accepted:

- **Base-load mode**: 6h at 350 MW, EUR 65/MWh (lower price, longer run)
- **Peak-shaving mode**: 4h at 400 MW, EUR 85/MWh (higher price, shorter run)

An `ExclusiveGroupBid` ensures EUPHEMIA can only accept one of these options.

In [ ]:
base_load_option = indivisible_block_bid(
    bidding_zone=BiddingZone.DE_LU,
    direction=Direction.SELL,
    delivery_period=base_load_period,
    price=Decimal("65.00"),
    volume=Decimal("350"),
    bid_id="borgholt-base-load",
    metadata={"mode": "base-load"},
)

peak_load_option = indivisible_block_bid(
    bidding_zone=BiddingZone.DE_LU,
    direction=Direction.SELL,
    delivery_period=peak_load_period,
    price=Decimal("85.00"),
    volume=Decimal("400"),
    bid_id="borgholt-peak-load",
    metadata={"mode": "peak-shaving"},
)

operating_mode_group = exclusive_group(
    block_bids=[base_load_option, peak_load_option],
    group_id="borgholt-operating-modes",
    metadata={"asset": "borgholt-ccgt", "description": "Mutually exclusive operating modes"},
)

print(f"Exclusive group: {operating_mode_group.group_id}")
print(f"  Members: {operating_mode_group.member_count}")
for b in operating_mode_group.block_bids:
    period = b.delivery_period
    start_str = period.start.astimezone(CET).strftime("%H:%M")
    end_str = period.end.astimezone(CET).strftime("%H:%M")
    print(f"    {b.bid_id}: {start_str}–{end_str}, {b.volume} MW @ {b.price} EUR/MWh")

## 5. Visualisation — Bid Timeline (Gantt Chart)

A Gantt-style chart shows the temporal relationship between bids. The arrow from
the startup bid to the main block illustrates the `LinkedBlockBid` dependency.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
fig.suptitle(
    "Borgholt CCGT — Block Bid Timeline (DE-LU, 2026-03-15)",
    fontsize=13, fontweight="bold"
)

def hour_offset(dt: datetime) -> float:
    """Convert a datetime to fractional hours from midnight CET."""
    local = dt.astimezone(CET)
    return local.hour + local.minute / 60

# Define bids to plot: (label, delivery_period, color, y_pos, annotation)
bid_bars = [
    ("Startup\n(Linked)", startup_linked.delivery_period, "#e67e22", 3,
     f"{startup_linked.price:.2f} EUR/MWh\n{startup_linked.volume} MW"),
    ("Main Block\n(Parent)", main_block.delivery_period, "#2980b9", 2,
     f"{main_block.price} EUR/MWh\n{main_block.volume} MW"),
    ("Base-load\n(Exclusive)", base_load_option.delivery_period, "#27ae60", 1,
     f"{base_load_option.price} EUR/MWh\n{base_load_option.volume} MW"),
    ("Peak-load\n(Exclusive)", peak_load_option.delivery_period, "#c0392b", 0,
     f"{peak_load_option.price} EUR/MWh\n{peak_load_option.volume} MW"),
]

bar_height = 0.5
for label, period, color, y, annotation in bid_bars:
    x_start = hour_offset(period.start)
    x_end = hour_offset(period.end)
    width = x_end - x_start
    ax.barh(y, width, left=x_start, height=bar_height,
            color=color, alpha=0.85, edgecolor="white", linewidth=1.5)
    ax.text(x_start + width / 2, y, annotation,
            ha="center", va="center", fontsize=8, color="white", fontweight="bold")
    ax.text(x_start - 0.1, y, label, ha="right", va="center", fontsize=9)

# Draw dependency arrow: startup → main block
ax.annotate(
    "", xy=(hour_offset(main_block.delivery_period.start), 2.25),
    xytext=(hour_offset(startup_linked.delivery_period.end), 2.75),
    arrowprops=dict(arrowstyle="->", color="black", lw=1.5),
)
ax.text(
    hour_offset(startup_linked.delivery_period.end) + 0.05, 2.5,
    "depends on", fontsize=8, va="center", color="black", style="italic"
)

# Exclusive group brace
ax.annotate(
    "Exclusive Group",
    xy=(8.0, 0.5), xytext=(5.5, 0.5),
    fontsize=9, color="#7f8c8d", style="italic",
    arrowprops=dict(arrowstyle="}-{", color="#7f8c8d", lw=1.5,
                    connectionstyle="arc3,rad=0"),
)

# Formatting
ax.set_xlim(5, 16)
ax.set_ylim(-0.6, 3.6)
ax.set_xlabel("Hour (CET)")
ax.set_xticks(range(5, 17))
ax.set_xticklabels([f"{h:02d}:00" for h in range(5, 17)])
ax.set_yticks([])
ax.grid(True, axis="x", alpha=0.3)
ax.spines["left"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)

plt.tight_layout()
plt.savefig("block_bid_timeline.png", dpi=120, bbox_inches="tight")
plt.show()
print("Figure saved: block_bid_timeline.png")

## 6. Revenue Analysis

Compare expected revenue for the two operating modes at a hypothetical market clearing price.

In [ ]:
# Hypothetical market clearing prices to evaluate
clearing_prices = [55, 65, 75, 85, 95, 110]

print(f"{'Clearing Price':>16} {'Base-load Revenue':>20} {'Peak Revenue':>16} {'Winner':>10}")
print("-" * 70)

for cp in clearing_prices:
    cp_d = Decimal(str(cp))

    # Base-load: 6h × 350 MW (only clears if cp >= 65)
    if cp_d >= base_load_option.price:
        base_hours = Decimal(str(base_load_option.delivery_period.mtu_count)) * Decimal("0.25")
        base_rev = cp_d * base_load_option.volume * base_hours
    else:
        base_rev = Decimal("0")

    # Peak: 4h × 400 MW (only clears if cp >= 85)
    if cp_d >= peak_load_option.price:
        peak_hours = Decimal(str(peak_load_option.delivery_period.mtu_count)) * Decimal("0.25")
        peak_rev = cp_d * peak_load_option.volume * peak_hours
    else:
        peak_rev = Decimal("0")

    winner = "BASE" if base_rev >= peak_rev else "PEAK"
    if base_rev == 0 and peak_rev == 0:
        winner = "NONE"

    print(f"  {cp:>12} EUR/MWh  {float(base_rev):>15,.0f} EUR  {float(peak_rev):>12,.0f} EUR  {winner:>10}")

## Summary

This notebook showed how to model a CCGT plant's bidding strategy using three block bid types:

- **`BlockBid` (indivisible)**: All-or-nothing 6-hour generation block
- **`LinkedBlockBid`**: Startup cost recovery that depends on the parent block being accepted
- **`ExclusiveGroupBid`**: Two mutually exclusive operating modes (base-load vs. peak-shaving)

Key design decisions:
- `min_acceptance_ratio=1.0` ensures the CCGT only runs if the full block clears
- Startup cost is embedded in the price via `EUR/MWh = marginal + (startup_cost / MWh)`
- The exclusive group lets the market choose the economically optimal mode

Next: see `03_merit_order_curves.ipynb` for aggregate portfolio curve construction.